In [ ]:
import pandas as pd
import requests
import cv2
import mediapipe as mp
import numpy as np
import csv
import json
from tqdm import tqdm
import os


In [ ]:
# Load your CSV
df = pd.read_csv('buddha_sculptureCC0_fig.csv')

In [ ]:
# Initialize Mediapipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)
mp_drawing = mp.solutions.drawing_utils

results_list = []

# Create a folder to store downloaded images
os.makedirs('sculpture_images', exist_ok=True)

for idx, row in tqdm(df.iterrows(), total=len(df)):
    object_id = row['Object ID']
    image_url = row['Link Image']

    if pd.isna(image_url) or not isinstance(image_url, str):
        continue  # Skip if no image

    # Download image
    img_path = f'sculpture_images/{object_id}.jpg'
    try:
        response = requests.get(image_url, timeout=10)
        with open(img_path, 'wb') as f:
            f.write(response.content)
    except Exception as e:
        print(f"Failed to download {image_url}: {e}")
        continue

    # Read image
    image = cv2.imread(img_path)
    if image is None:
        print(f"Failed to read {img_path}")
        continue

    # Convert BGR to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Run pose detection
    result = pose.process(image_rgb)

    if not result.pose_landmarks:
        print(f"No pose detected for {object_id}")
        continue

    # Extract keypoints
    keypoints = []
    for idx_lm, lm in enumerate(result.pose_landmarks.landmark):
        keypoints.append({
            "name": mp_pose.PoseLandmark(idx_lm).name,
            "x": lm.x,
            "y": lm.y,
            "z": lm.z,
            "visibility": lm.visibility
        })

    # Save result
    results_list.append({
        "object_id": object_id,
        "image_url": image_url,
        "keypoints": keypoints
    })




In [ ]:
# Save to JSON
with open('pose_results.json', 'w') as f:
    json.dump(results_list, f, indent=2)

print("✅ All pose data saved to pose_results.json")

In [ ]:
pose_df = pd.read_json('pose_results.json')
pose_df.info()